In [12]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [13]:
ds = load_dataset("christinacdl/binary_hate_speech")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3883 non-null   object
 1   label   3883 non-null   object
dtypes: object(2)
memory usage: 60.8+ KB


In [14]:
test['label'] = test['label'].apply(lambda x: 'hateful' if x == 'OFF_HATEFUL_TOXIC' else 'safe')

labels = test['label'].unique()

test

,text,label
0,i have to study... #face #pizza (i stole my ...,safe
1,days porn movie srilankanboyssex,safe
2,feeling for friends left in the place we use...,safe
3,why only target little #muslim children for mi...,hateful
4,M. Todd Henderson dared compare SCOTUS nominee...,safe
...,...,...
3878,Maybe he just thought you looked like a sand n...,hateful
3879,ATTENTION SYRIAN 'REFUGEES'! You can go home n...,safe
3880,A hoe wants attention a women wants respect.,hateful
3881,@user there is only one requirement for the jo...,hateful


In [15]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_14112\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


53637120

In [16]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [17]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "deepseek-r1:1.5b",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of possibly hateful content. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Tweet: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()
    content = content.split('</think>')[1]

    if 'safe' in content:
        content = 'safe'
    elif 'hateful' in content:
        content = 'hateful'
    else:
        content = 'error'

    print(f'Tweet: {text}')
    print(f'Classification: {content}')

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [19]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_14112\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Tweet: i have to study...   #face #pizza (i stole my roommate's idea for taking this pic) 
Classification: hateful
Tweet:   days porn movie srilankanboyssex  
Classification: safe
Tweet: feeling   for friends left in the place we use to live and raised our #son. life you live in the #mining world! 
Classification: hateful
Tweet: why only target little #muslim children for mispronouncing "terraced house" &amp; treat them like criminals!â¦ 
Classification: hateful
Tweet: M. Todd Henderson dared compare SCOTUS nominee Brett Kavanaugh with Sonia Sotomayor. They called him a racist and worse. http://www.patheos.com/blogs/dispatches/2018/08/09...
Classification: safe
Tweet: @user can someone please explain to these liberals that it takes more than clapping and kneeling to actually make a real difference 
Classification: safe
Tweet: Italy is the world’s 9th economy and is on a trajectory that in the long run will end in an economic implosion due to demographic decline , not solvable by force

C:\Users\Rafael\AppData\Local\Temp\ipykernel_14112\2610005304.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)
C:\Users\Rafael\AppData\Local\Temp\ipykernel_14112\2610005304.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply

In [20]:
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
0,i have to study... #face #pizza (i stole my ...,safe,hateful,6.039485,2426,87.238281,4.013220
1,days porn movie srilankanboyssex,safe,safe,3.678696,2411,87.695312,1.638991
2,feeling for friends left in the place we use...,safe,hateful,3.330060,2411,88.191406,1.286550
3,why only target little #muslim children for mi...,hateful,hateful,3.472393,2436,87.847656,1.434300
4,M. Todd Henderson dared compare SCOTUS nominee...,safe,safe,5.168773,2434,88.414062,3.112754
...,...,...,...,...,...,...,...
83,@USER I'M ALMOST SHOT BY LA GANG MEMBER/ILLEGA...,hateful,hateful,4.197741,2486,89.144531,2.151505
85,RT @toddknife: Full @weakenednachos set (excep...,hateful,safe,3.765255,2451,89.015625,1.730010
86,@user @user @user Tell this idiot to try with ...,hateful,safe,4.108407,2444,89.027344,2.062533
88,#blacklivesmatter that's why bilal called the ...,hateful,safe,3.668499,2433,89.078125,1.642310


In [21]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.635135
F1 score: 0.640280
Precision: 0.654762
Recall: 0.635135


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 4.214832315573821
Average VRAM usage: 2440.4054054054054
Average RAM usage: 88.86782094594595
Average total time: 2.1731648297297297


In [23]:
# save results to txt
with open('results/deepseekR1_ZS_binary1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')